# Import Core Libraries
Load NumPy and pandas for numerical work and dataframe handling.

In [45]:
import numpy as np
import pandas as pd


# Import Machine Learning Tools
Bring in the scikit-learn utilities needed for splitting data, preprocessing, feature selection, modeling, and building the pipeline.

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest , chi2


# Load the Titanic Dataset
Read the training data from the datasets folder and preview a few random rows.

In [47]:
df = pd.read_csv("../../datasets/train.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
830,831,1,3,"Yasbeck, Mrs. Antoni (Selini Alexander)",female,15.00,1,0,2659,14.4542,NaN,C
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.00,1,5,347082,31.2750,NaN,S
748,749,0,1,"Marvin, Mr. Daniel Warner",male,19.00,1,0,113773,53.1000,D30,S
97,98,1,1,"Greenfield, Mr. William Bertram",male,23.00,0,1,PC 17759,63.3583,D10 D12,C
469,470,1,3,"Baclini, Miss. Helene Barbara",female,0.75,2,1,2666,19.2583,NaN,C


# Remove Unused Columns
Drop identifier and high-cardinality text columns that are not used in this simple pipeline.

In [48]:

df = df.drop(columns = ["Cabin", "PassengerId" , "Name" , "Ticket"])

# Check Missing Values
Inspect missing values so the preprocessing steps can handle them correctly.

In [49]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

# Split Features and Target
Separate the input features from the survival target and create training and test sets.

In [50]:
X_train , X_test , Y_train , Y_test = train_test_split(df.drop(columns = ["Survived"]) , df["Survived"] , test_size = 0.2 , random_state = 42)

# Preview Training Features
Look at a sample of the feature data that will be passed into the pipeline.

In [51]:
X_train.sample(5)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
562,2,male,28.0,0,0,13.5000,S
380,1,female,42.0,0,0,227.5250,C
677,3,female,18.0,0,0,9.8417,S
788,3,male,1.0,1,2,20.5750,S
55,1,male,NaN,0,0,35.5000,S


# Preview Training Labels
Check a sample of the target values used for training.

In [52]:
Y_train.sample(5)

706    1
795    0
197    0
534    0
167    0
Name: Survived, dtype: int64

# Review Feature Columns
Confirm the remaining column order before applying column-based transformations.

In [53]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

# Impute Missing Values
Fill missing Age values and replace missing Embarked values with the most frequent category.

In [54]:
transformer1 = ColumnTransformer(transformers=[
    ("impute_age" , SimpleImputer() , [2]),
    ("impute_embarked" , SimpleImputer(strategy= "most_frequent") , [6])
]
,remainder= "passthrough"
)

# Encode Categorical Features
Apply one-hot encoding to categorical columns while keeping the remaining features for later steps.

In [55]:
transformer2 = ColumnTransformer(transformers=[
    ("ohe_sex_embarked", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), [1, 6])
], remainder="passthrough")

# Scale Numeric Features
Normalize the transformed features so they are on a comparable range for chi-square feature selection.

In [56]:
transformer3 = ColumnTransformer(transformers = [
    ("scaling" , MinMaxScaler()  , slice(0,10))
])

# Select Best Features
Keep the most useful features according to the chi-square score.

In [57]:
transformer4 = SelectKBest(score_func=chi2, k = 8)#this is feature selection

# Create the Model
Use a decision tree classifier as the final estimator in the pipeline.

In [58]:
transformer5  = DecisionTreeClassifier()

# Build the Pipeline
Chain preprocessing, feature selection, and the classifier into one reusable pipeline.

In [59]:
pipe = make_pipeline(transformer1,transformer2,transformer3,transformer4,transformer5)#now this will make the 
#pipeline

# Train the Pipeline
Fit the complete pipeline on the training data.

In [60]:
pipe.fit(X_train , Y_train)

,steps,"[('columntransformer-1', ...), ('columntransformer-2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Pipeline Note
Mark this section as the complete end-to-end pipeline.

In [61]:
#this is the whole pipeline

# Inspect Pipeline Steps
Display the named steps inside the fitted pipeline.

In [62]:
pipe.named_steps["columntransformer-2"]

,transformers,"[('ohe_sex_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


# Make Predictions
Use the fitted pipeline to predict survival on the test set.

In [63]:
Y_pred = pipe.predict(X_test)

# Evaluate Pipeline Accuracy
Calculate the percentage accuracy of the pipeline predictions.

In [64]:
#for printing the prediction accuracy
from sklearn.metrics import accuracy_score
accuracy_score(Y_test , Y_pred)*100


62.56983240223464